# Person 2 — Pretrained ASR baselines (Icelandic, Samrómur 21.05 v1)

These runs are **baselines**: published checkpoints with **no training by our group**.

| Key | Checkpoint | Note |
|---|---|---|
| `whisper_small` | `openai/whisper-small` | Multilingual, zero-shot, `language="is"` |
| `wav2vec2_is` | `language-and-voice-lab/wav2vec2-large-xlsr-53-icelandic-ep30-967h` | wav2vec 2.0 + CTC head, trained on Samrómur Milljón (possible test overlap) |
| `whisper_small_is` | `Valdimarb13/whisper-small-icelandic` | Whisper small fine-tuned on `samromur_asr` by a third party |

Protocol: settings (Whisper beam size) are chosen on **dev**. **Test** runs once at the end with the frozen setting.
All audio comes from the repository via Git LFS, so there are no manual uploads. Runtime: **GPU**. Colab: Runtime → Change runtime type → T4. Kaggle: Settings → Accelerator → GPU T4 x2, and Settings → Internet → On.

In [ ]:
!nvidia-smi

## 1. Get the repository and the dev/test audio

In [ ]:
import os
WORK = "/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "/content"   # Colab also has a /kaggle folder
REPO_DIR = f"{WORK}/icelandic-asr-project3"

# Fixed commits, so a rerun uses exactly the data and code behind the reported results.
DATA_REPO = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
DATA_COMMIT = "ef4d5156cb599348dd9dc55934ff391cc8c5ee5d"   # frozen v1 audio, manifests and references
CODE_REPO = "https://github.com/MuhammetGoksenin/icelandic-asr-project3.git"
CODE_COMMIT = "d830083bb3c7334b64a1bf8c256b2dac7206db8b"                             # baselines/, training/, tests, requirements-modeling.txt
CODE_PATHS = "baselines training tests/test_pretrained.py tests/test_hubert_ctc.py requirements-modeling.txt"
assert len(CODE_COMMIT) == 40, "pin CODE_COMMIT to a full commit hash"

# Kaggle images ship without git-lfs
!which git-lfs >/dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs)
!git lfs install
if not os.path.exists(REPO_DIR):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q {DATA_REPO} {REPO_DIR}
%cd {REPO_DIR}
!GIT_LFS_SKIP_SMUDGE=1 git checkout -q {DATA_COMMIT}
!git lfs pull --include="data/cleaned/samromur-21.05-20h-v1/dev/**,data/cleaned/samromur-21.05-20h-v1/test/**" --exclude=""
!GIT_LFS_SKIP_SMUDGE=1 git fetch -q {CODE_REPO} {CODE_COMMIT}
!git checkout {CODE_COMMIT} -- {CODE_PATHS}
!echo "Data commit: $(git rev-parse HEAD)"; echo "Code commit: {CODE_COMMIT}"

## 2. Install pinned packages
`requirements-modeling.txt` pins PyTorch, Transformers, Accelerate and the rest; versions are also recorded in every `model_info` JSON.

In [ ]:
!pip install -q -r requirements-modeling.txt   # pinned modeling environment; keeps Kaggle's torch 2.11.0+cu130
# discover -s tests: Kaggle has an unrelated 'tests' package in site-packages
!python -m unittest discover -s tests -p "test_pretrained.py" -v

## 3. Smoke test (first 20 dev clips per model)

In [ ]:
for model in ["whisper_small", "wav2vec2_is", "whisper_small_is"]:
    !python baselines/run_pretrained.py --model {model} --split dev --limit 20 --num-beams 1

## 4. Dev runs: choose Whisper beam size
wav2vec 2.0 uses greedy CTC decoding, so it has no beam to choose.

In [ ]:
!python baselines/run_pretrained.py --model whisper_small    --split dev --num-beams 1 5
!python baselines/run_pretrained.py --model whisper_small_is --split dev --num-beams 1 5
!python baselines/run_pretrained.py --model wav2vec2_is      --split dev

In [ ]:
import json, glob
import pandas as pd

def summary(split):
    rows = []
    for path in sorted(glob.glob(f"results/pretrained/model_info/*_{split}*.json")):
        if "_first" in path:
            continue
        d = json.load(open(path, encoding="utf-8"))
        rows.append({"run": os.path.basename(path)[:-5], "checkpoint": d["checkpoint"],
                     "revision": d["revision"][:10], "beams": d.get("num_beams", "-"),
                     "WER": round(d["scores_preliminary"]["wer"] * 100, 2),
                     "CER": round(d["scores_preliminary"]["cer"] * 100, 2),
                     "RTF": d["rtf_incl_audio_loading"], "peak GPU GB": d["peak_gpu_memory_gb"]})
    return pd.DataFrame(rows)

summary("dev")

## 5. Freeze settings, then run test once
Set each beam to whatever had the lower **dev** WER above, and write the choice in the README.

In [ ]:
BEAM = {"whisper_small": 5, "whisper_small_is": 5}   # <- set from the dev table

!python baselines/run_pretrained.py --model whisper_small    --split test --num-beams {BEAM["whisper_small"]}
!python baselines/run_pretrained.py --model whisper_small_is --split test --num-beams {BEAM["whisper_small_is"]}
!python baselines/run_pretrained.py --model wav2vec2_is      --split test
summary("test")

## 6. Save the environment and package the results
Download the zip and commit `results/pretrained/` **from your own GitHub account**.

In [ ]:
!pip freeze > results/pretrained/colab_pip_freeze.txt
!rm -f results/pretrained/predictions/*_first20.csv results/pretrained/model_info/*_first20.json
!zip -qr {WORK}/pretrained_results.zip results/pretrained
if WORK == "/kaggle/working":   # Kaggle also ships google.colab, but its download() fails there
    print(f"Kaggle: download {WORK}/pretrained_results.zip from the Output panel on the right.")
else:
    from google.colab import files
    files.download(f"{WORK}/pretrained_results.zip")